# SqueezeNet 1.1 on EuroSAT RGB

Train and evaluate SqueezeNet using the shared `eurosat_splits.csv` and
`eurosat_classes.json`. This notebook preserves the existing split assignments,
64 x 64 preprocessing, two training stages, and result exports.

Install `requirements.txt`, open this notebook with a Python kernel in the
repository folder, edit the configuration below, and run the cells in order.
The first run downloads ImageNet weights; leaving `data_dir=None` also downloads
the EuroSAT dataset through KaggleHub. Downloads require internet access.
Training uses CUDA when available, otherwise the CPU.

Restart the kernel and run all cells for a fresh experiment. Re-running a training
cell continues with the current in-memory model and history. Outputs are saved
to `output_dir`; choose a new directory to keep previous experiment results.


## Configuration

Set `data_dir` to `Path("C:/datasets/EuroSAT")` if the images are already available;
that directory must contain the ten class folders. Paths below are relative to
the repository folder. Set `ROOT` explicitly if your kernel starts elsewhere.

The defaults train the classifier for 5 epochs and fine-tune for 20. For a shorter
trial, set both epoch counts to 1; this still uses every image in the manifest.
`num_workers=0` supports dataset classes defined in a notebook, including on
Windows. `show_plots=True` displays charts as well as saving them.


In [ ]:
from pathlib import Path
from types import SimpleNamespace

ROOT = Path.cwd().resolve()
if not (ROOT / "eurosat_data.py").is_file():
    raise FileNotFoundError("Set ROOT to the repository folder containing eurosat_data.py")

# Keep the shared helper importable if ROOT is set explicitly above.
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

args = SimpleNamespace(
    data_dir=None,  # Or Path("C:/datasets/EuroSAT")
    splits=ROOT / "eurosat_splits.csv",
    classes=ROOT / "eurosat_classes.json",
    output_dir=ROOT / "squeezenet_results",
    seed=42,
    batch_size=64,
    num_workers=0,
    head_epochs=5,
    fine_tune_epochs=20,
    show_plots=True,
)

for name in ("batch_size", "head_epochs", "fine_tune_epochs"):
    value = getattr(args, name)
    if not isinstance(value, int) or isinstance(value, bool) or value < 1:
        raise ValueError(f"{name} must be a positive integer")
if not isinstance(args.num_workers, int) or isinstance(args.num_workers, bool) or args.num_workers < 0:
    raise ValueError("num_workers must be a nonnegative integer")


## Imports and reproducibility

Seed the random generators, select the device, and create the output folder.


In [ ]:
import json
import time
import random

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
if not args.show_plots:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import (
    squeezenet1_1,
    SqueezeNet1_1_Weights
)

from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


SEED = args.seed

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUTPUT_DIR = args.output_dir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", device)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
